In [42]:
from datasets import Dataset
import pandas as pd 

In [43]:
toxic_train = pd.read_csv(r'..\Data\toxic\train.csv')
pprint(type(toxic_train.columns))
toxic_train.sample(5)

<class 'pandas.Index'>


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
65678,afb10b4f5c271324,Notability of Phillip Winn\nA tag has been pla...,0,0,0,0,0,0
113662,5fcabbc9454b64db,"I like the article in many respects, consideri...",0,0,0,0,0,0
137460,df8de9a6700e84d4,"""\n\n Your warning concerning LittleGreenRoset...",0,0,0,0,0,0
148580,4ef5c81065f4f46f,That is both remarkably considerate and very f...,0,0,0,0,0,0
42507,716e3ebcde2e6b27,Careful there - the combination of straw men a...,0,0,0,0,0,0


In [44]:
label_cols = toxic_train.columns.difference(["id","comment_text"])
label_cols = list(label_cols)

In [45]:
toxic_train["toxic"] = toxic_train[label_cols].max(axis=1) 

In [46]:
jigsaw_train = toxic_train[["comment_text","toxic"]].rename(columns={"comment_text":"text"})
jigsaw_train.sample()

,text,toxic
154418,"""\nHey Martin. Spoondfeeding required! Where e...",0


In [47]:
toxic_test = pd.read_csv(r'..\Data\toxic\test.csv')
test_labels = pd.read_csv(r'..\Data\toxic\test_labels.csv')
test =  toxic_test.merge(test_labels,on="id")
test = test[test["toxic"]!=-1]
test["toxic"] = test[label_cols].max(axis=1)
jigsaw_test = test[["comment_text","toxic"]].rename(columns={"comment_text":"text"})


In [48]:
jigsaw_test.sample(5)

,text,toxic
110778,""":::::I apologize; I'm a journalist, and I sho...",0
119812,""" \n\n == Jimmy... == \n\n I just wanted to co...",0
14078,==Thanks== \n I'd just copied another disambig...,0
16500,== Why all muslims should be burned alive == \...,1
35065,""" \n\n \n ==Administrator's Noticeboard case ...",0


In [49]:
hate_meta = pd.read_csv(r'..\Data\hate\annotations_metadata.csv')
hate_meta.sample(5)

,file_id,user_id,subforum_id,num_contexts,label
2430,13552491_3,576755,1346,0,hate
5977,14037278_1,578715,1387,0,hate
1732,14302429_2,576731,1380,0,noHate
1273,13472253_4,572067,1346,2,noHate
10266,14070890_1,572082,1383,0,noHate


In [50]:
hate_meta["label"] =hate_meta['label'].isin(["hate"])
hate_meta.sample(5)

,file_id,user_id,subforum_id,num_contexts,label
1436,13850034_1,592500,1391,0,False
1484,13866450_2,591103,1391,0,False
876,30558474_1,576721,1348,0,False
251,13592269_1,572266,1393,0,False
5297,30602452_1,577073,1354,0,False


In [51]:
hate_meta["toxic"] = hate_meta['label'].astype(int)
hate_meta.sample(5)

,file_id,user_id,subforum_id,num_contexts,label,toxic
9705,31706302_5,586694,1363,0,False,0
6910,13607285_1,589508,1393,0,False,0
145,13456696_5,572219,1396,2,False,0
6409,30571340_1,575662,1359,0,False,0
10359,14263956_1,573514,1380,0,False,0


In [52]:
texts=[]
for fid in hate_meta['file_id']:
  with open(f"../Data/hate/all_files/{fid}.txt",encoding="utf-8",errors="replace") as f:
    texts.append(f.read().strip())
hate_meta["text"] = texts
hate = hate_meta[["text","toxic"]]
hate.sample(5)

,text,toxic
4067,I do n't look down on these degrees as much as...,0
9859,They're trying to turn the Internet into a mir...,0
1906,Most enlightening .,0
1323,I guess the term `` blond '' is somewhat subje...,0
6341,Use your guns to fight for THE WHITE RACE !!,0


In [53]:
combined = pd.concat([jigsaw_test,jigsaw_train,hate],ignore_index=True)
combined.sample(5)

,text,toxic
8491,"Done. Thank you. I'm rather new at this, thoug...",0
172545,Merry Merry\nTo you and yours,0
69465,"No, Melanie, I do not think it is a consensus ...",0
160559,"""\n\n Re:""""Lord"""" Sekule Drljevic \nThe war cr...",0
12228,"REDIRECT Talk:Marine Protection, Research, and...",0


In [56]:
dataset = Dataset.from_pandas(combined)
dataset = dataset.shuffle(seed=45)
dataset = dataset.train_test_split(test_size=0.2,seed=42)
dataset.save_to_disk(r"..\Data\combined_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 46899/46899 [00:00<00:00, 364331.30 examples/s]
